In [0]:
catalog = "flights_project"

df_silver = spark.table(f"{catalog}.silver.flights_clean")
display(df_silver.limit(5))

flight_date,OP_UNIQUE_CARRIER,TAIL_NUM,OP_CARRIER_FL_NUM,ORIGIN,ORIGIN_CITY_NAME,ORIGIN_STATE_ABR,DEST,DEST_CITY_NAME,DEST_STATE_ABR,dep_hour,day_of_week,is_weekend,dep_delay,arr_delay,flight_status,distance,air_time,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay,_ingested_at,_source_file
2025-07-01,AA,N104NN,2,LAX,"Los Angeles, CA",CA,JFK,"New York, NY",NY,7,3,false,44.0,84.0,DELAYED,2475.0,341.0,0.0,44.0,40.0,0.0,0.0,2026-08-16T18:50:54.883Z,/Volumes/flights_project/bronze/raw_landing/flights_2025_07.csv
2025-07-01,AA,N200NV,2271,MIA,"Miami, FL",FL,LGA,"New York, NY",NY,17,3,false,194.0,230.0,DELAYED,1096.0,196.0,0.0,0.0,230.0,0.0,0.0,2026-08-16T18:50:54.883Z,/Volumes/flights_project/bronze/raw_landing/flights_2025_07.csv
2025-07-01,AA,N301NW,1583,PIT,"Pittsburgh, PA",PA,DFW,"Dallas/Fort Worth, TX",TX,12,3,false,23.0,29.0,DELAYED,1067.0,150.0,11.0,0.0,6.0,0.0,12.0,2026-08-16T18:50:54.883Z,/Volumes/flights_project/bronze/raw_landing/flights_2025_07.csv
2025-07-01,AA,N339TP,2943,DEN,"Denver, CO",CO,PHL,"Philadelphia, PA",PA,14,3,false,94.0,73.0,DELAYED,1558.0,183.0,8.0,0.0,0.0,0.0,65.0,2026-08-16T18:50:54.883Z,/Volumes/flights_project/bronze/raw_landing/flights_2025_07.csv
2025-07-01,AA,N402AN,407,ANC,"Anchorage, AK",AK,DFW,"Dallas/Fort Worth, TX",TX,6,3,false,-8.0,-24.0,ON_TIME,3043.0,357.0,null,null,null,null,null,2026-08-16T18:50:54.883Z,/Volumes/flights_project/bronze/raw_landing/flights_2025_07.csv


In [0]:
from pyspark.sql.functions import count, avg, sum as _sum, when, col, round as _round

gold_carrier = (df_silver
    .groupBy("OP_UNIQUE_CARRIER")
    .agg(
        count("*").alias("total_flights"),
        _sum(when(col("flight_status") == "ON_TIME", 1).otherwise(0)).alias("on_time_flights"),
        _sum(when(col("flight_status") == "CANCELLED", 1).otherwise(0)).alias("cancelled_flights"),
        _round(avg("dep_delay"), 2).alias("avg_dep_delay_min"),
        _round(avg("arr_delay"), 2).alias("avg_arr_delay_min"),
    )
    .withColumn("on_time_pct", _round(col("on_time_flights") / col("total_flights") * 100, 2))
    .withColumn("cancellation_pct", _round(col("cancelled_flights") / col("total_flights") * 100, 2))
    .orderBy(col("on_time_pct").desc())
)

(gold_carrier.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.gold.delay_by_carrier"))

display(gold_carrier)

OP_UNIQUE_CARRIER,total_flights,on_time_flights,cancelled_flights,avg_dep_delay_min,avg_arr_delay_min,on_time_pct,cancellation_pct
HA,26931,22504,285,8.33,7.48,83.56,1.06
OO,282159,228483,3178,11.6,8.01,80.98,1.13
UA,267466,213493,2877,12.42,7.52,79.82,1.08
DL,344657,274904,4216,11.6,5.89,79.76,1.22
YX,116989,93045,4625,8.88,5.42,79.53,3.95
MQ,101070,79970,2404,9.45,6.52,79.12,2.38
NK,67282,52995,1138,10.87,4.62,78.77,1.69
AS,81444,63774,1260,8.17,4.52,78.3,1.55
G4,44719,34072,196,14.89,12.37,76.19,0.44
WN,470194,348364,5514,13.08,6.47,74.09,1.17


In [0]:
gold_airport_hour = (df_silver
    .groupBy("ORIGIN", "ORIGIN_CITY_NAME", "dep_hour")
    .agg(
        count("*").alias("total_flights"),
        _round(avg("dep_delay"), 2).alias("avg_dep_delay_min"),
        _sum(when(col("flight_status") == "DELAYED", 1).otherwise(0)).alias("delayed_flights"),
    )
    .withColumn("delay_rate_pct", _round(col("delayed_flights") / col("total_flights") * 100, 2))
    .filter(col("total_flights") >= 20)  # drop noisy low-volume slots
    .orderBy(col("delay_rate_pct").desc())
)

(gold_airport_hour.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.gold.delay_by_airport_hour"))

display(gold_airport_hour.limit(20))

ORIGIN,ORIGIN_CITY_NAME,dep_hour,total_flights,avg_dep_delay_min,delayed_flights,delay_rate_pct
TOL,"Toledo, OH",21,33,65.21,29,87.88
SIT,"Sitka, AK",13,34,33.32,25,73.53
TRI,"Bristol/Johnson City/Kingsport, TN",15,24,38.08,17,70.83
ROA,"Roanoke, VA",21,25,68.36,16,64.0
EVV,"Evansville, IN",20,22,109.41,14,63.64
LCK,"Columbus, OH",21,38,87.08,24,63.16
HGR,"Hagerstown, MD",21,24,41.0,15,62.5
TYS,"Knoxville, TN",21,70,36.69,43,61.43
SGF,"Springfield, MO",20,49,57.06,30,61.22
RST,"Rochester, MN",14,27,40.96,16,59.26


In [0]:
gold_delay_causes = (df_silver
    .filter(col("flight_status") == "DELAYED")
    .agg(
        _round(avg("carrier_delay"), 2).alias("avg_carrier_delay"),
        _round(avg("weather_delay"), 2).alias("avg_weather_delay"),
        _round(avg("nas_delay"), 2).alias("avg_nas_delay"),
        _round(avg("security_delay"), 2).alias("avg_security_delay"),
        _round(avg("late_aircraft_delay"), 2).alias("avg_late_aircraft_delay"),
    )
)

(gold_delay_causes.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.gold.delay_causes_breakdown"))

display(gold_delay_causes)

avg_carrier_delay,avg_weather_delay,avg_nas_delay,avg_security_delay,avg_late_aircraft_delay
29.25,5.86,13.31,0.12,35.77


In [0]:
gold_route_risk = (df_silver
    .groupBy("ORIGIN", "DEST", "OP_UNIQUE_CARRIER")
    .agg(
        count("*").alias("total_flights"),
        _round(avg("dep_delay"), 2).alias("avg_dep_delay_min"),
        _sum(when(col("flight_status") == "DELAYED", 1).otherwise(0)).alias("delayed_flights"),
    )
    .withColumn("delay_rate_pct", _round(col("delayed_flights") / col("total_flights") * 100, 2))
    .filter(col("total_flights") >= 30)
    .orderBy(col("delay_rate_pct").desc())
)

(gold_route_risk.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.gold.route_risk_features"))

display(gold_route_risk.limit(20))

ORIGIN,DEST,OP_UNIQUE_CARRIER,total_flights,avg_dep_delay_min,delayed_flights,delay_rate_pct
SAN,ORD,F9,31,82.19,24,77.42
BOS,PDX,B6,32,75.65,24,75.0
TOL,SFB,G4,43,56.51,32,74.42
SEA,DAL,WN,36,27.39,26,72.22
MEM,SFB,G4,38,62.82,27,71.05
CKB,SFB,G4,40,174.85,28,70.0
ROA,SFB,G4,45,69.45,31,68.89
ANC,ATL,DL,35,36.23,24,68.57
SFO,AUS,WN,31,25.87,21,67.74
ANC,SEA,HA,63,55.31,42,66.67


In [0]:
%sql
SELECT 'delay_by_carrier' AS tbl, COUNT(*) AS rows FROM flights_project.gold.delay_by_carrier
UNION ALL
SELECT 'delay_by_airport_hour', COUNT(*) FROM flights_project.gold.delay_by_airport_hour
UNION ALL
SELECT 'delay_causes_breakdown', COUNT(*) FROM flights_project.gold.delay_causes_breakdown
UNION ALL
SELECT 'route_risk_features', COUNT(*) FROM flights_project.gold.route_risk_features;

tbl,rows
delay_by_carrier,14
delay_by_airport_hour,3625
delay_causes_breakdown,1
route_risk_features,9629


In [0]:
%sql
COMMENT ON TABLE flights_project.bronze.flights_raw IS 
  'Raw ingested US domestic flight records from BTS TranStats, landed via Autoloader with schema evolution. No transformations applied.';

COMMENT ON TABLE flights_project.silver.flights_clean IS 
  'Cleaned, typed, deduplicated flight records with derived flight_status, dep_hour, and weekend flags.';

COMMENT ON TABLE flights_project.gold.delay_by_carrier IS 
  'On-time performance and cancellation metrics aggregated by airline, refreshed from silver.flights_clean.';

COMMENT ON TABLE flights_project.gold.delay_by_airport_hour IS 
  'Delay risk by origin airport and departure hour, filtered to slots with 20+ flights for statistical reliability.';

COMMENT ON TABLE flights_project.gold.delay_causes_breakdown IS 
  'Average delay minutes attributed to carrier, weather, NAS, security, and late-aircraft causes, across delayed flights only.';

COMMENT ON TABLE flights_project.gold.route_risk_features IS 
  'Delay risk by origin-destination-carrier route combination, filtered to routes with 30+ flights.';

In [0]:
%sql
GRANT SELECT ON SCHEMA flights_project.gold TO `rohanmahendrauni28@gmail.com`;